In [1]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchaudio
import torchaudio.transforms as T
import torchaudio.functional as F
from tqdm.auto import tqdm

In [2]:
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [3]:
TRAIN_CSV = "Russian Emotional Speech/train.csv"
TEST_CSV = "Russian Emotional Speech/test.csv"
SAMPLE_RATE = 16000
DURATION = 3.0  # секунды, фиксированная длина аудио
N_MFCC = 40
N_MELS = 128
HOP_LENGTH = 512
N_FFT = 2048

BATCH_SIZE = 32
EPOCHS = 30
LEARNING_RATE = 1e-3

In [4]:
# Загрузка тренировочного CSV
train_df = pd.read_csv(TRAIN_CSV)  # колонки: name, path, emotion, text

In [41]:
EMOTIONS = ['anger', 'disgust', 'fear', 'enthusiasm', 'happiness', 'neutral', 'sadness']

emotion_to_label = {
    'anger': 0,
    'disgust': 1,
    'fear': 2,
    'enthusiasm': 3,
    'happiness': 4,
    'neutral': 5,
    'sadness': 6,
}

train_df = train_df[train_df['emotion'].isin(EMOTIONS)]

In [42]:
train_df

,name,path,emotion,text,label
0,32_happiness_enthusiasm_h_120,happiness_enthusiasm_32/32_happiness_enthusias...,happiness,"Конечно, расскажу, обязательно. Ой, сейчас рас...",4
1,36_disgust_happiness_d_130,disgust_happiness_36/36_disgust_happiness_d_13...,disgust,Вы ещё и профессию решили поменять.,1
2,34_anger_fear_a_060,anger_fear_34/34_anger_fear_a_060.wav,anger,"Ты знаешь, чем это для тебя закончится?",0
3,25_anger_disgust_a_010,anger_disgust_25/25_anger_disgust_a_010.wav,anger,Добрый день. Вы хотели бы приобрести недвижимо...,0
4,17_neutral_disgust_d_092,neutral_disgust_17/17_neutral_disgust_d_092.wav,disgust,"все ваши рекламные акции, пожалуйста, больше н...",1
...,...,...,...,...,...
1111,42_anger_fear_a_120,anger_fear_42/42_anger_fear_a_120.wav,anger,У вас там вообще всё нормально в квартире? Чё ...,0
1112,21_happiness_anger_a_120,happiness_anger_21/21_happiness_anger_a_120.wav,anger,Какао с молоком я не заказывала!,0
1113,45_happiness_sadness_s_142,happiness_sadness_45/45_happiness_sadness_s_14...,sadness,Ты злорадствуешь мне,6
1114,18_happiness_neutral_n_140,happiness_neutral_18/18_happiness_neutral_n_14...,neutral,"Считаю, что у тебя все получится, раз ты уже с...",5


In [49]:
# Кодирование меток
train_df['label'] = [int(emotion_to_label[train_df['emotion'].values[i]]) for i in range(len(train_df))]

In [50]:
train_df

,name,path,emotion,text,label
0,32_happiness_enthusiasm_h_120,happiness_enthusiasm_32/32_happiness_enthusias...,happiness,"Конечно, расскажу, обязательно. Ой, сейчас рас...",4
1,36_disgust_happiness_d_130,disgust_happiness_36/36_disgust_happiness_d_13...,disgust,Вы ещё и профессию решили поменять.,1
2,34_anger_fear_a_060,anger_fear_34/34_anger_fear_a_060.wav,anger,"Ты знаешь, чем это для тебя закончится?",0
3,25_anger_disgust_a_010,anger_disgust_25/25_anger_disgust_a_010.wav,anger,Добрый день. Вы хотели бы приобрести недвижимо...,0
4,17_neutral_disgust_d_092,neutral_disgust_17/17_neutral_disgust_d_092.wav,disgust,"все ваши рекламные акции, пожалуйста, больше н...",1
...,...,...,...,...,...
1111,42_anger_fear_a_120,anger_fear_42/42_anger_fear_a_120.wav,anger,У вас там вообще всё нормально в квартире? Чё ...,0
1112,21_happiness_anger_a_120,happiness_anger_21/21_happiness_anger_a_120.wav,anger,Какао с молоком я не заказывала!,0
1113,45_happiness_sadness_s_142,happiness_sadness_45/45_happiness_sadness_s_14...,sadness,Ты злорадствуешь мне,6
1114,18_happiness_neutral_n_140,happiness_neutral_18/18_happiness_neutral_n_14...,neutral,"Считаю, что у тебя все получится, раз ты уже с...",5


In [51]:
class EmotionDatasetRaw(Dataset):
    def __init__(self, df, audio_dir, target_length):
        self.df = df.reset_index(drop=True)
        self.audio_dir = audio_dir
        self.target_length = target_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        file_path = os.path.join(self.audio_dir, row['path'])
        waveform, sr = torchaudio.load(file_path)
        # Приводим к моно, если стерео
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)
        # Приведение к целевой длине
        if waveform.shape[1] < self.target_length:
            # pad
            pad_size = self.target_length - waveform.shape[1]
            waveform = torch.nn.functional.pad(waveform, (0, pad_size))
        else:
            waveform = waveform[:, :self.target_length]
        label = torch.tensor(row['label'], dtype=torch.long)
        return waveform, label

In [52]:
dataset = EmotionDatasetRaw(train_df, 'Russian Emotional Speech/train', int(SAMPLE_RATE * DURATION))

In [53]:
len(dataset)

1116

In [54]:
# Разделение на train/val
from sklearn.model_selection import train_test_split
train_idx, val_idx = train_test_split(range(len(train_df)), test_size=0.2,
                                      random_state=RANDOM_SEED, stratify=train_df['label'])
train_dataset = EmotionDatasetRaw(train_df.iloc[train_idx], 'Russian Emotional Speech/train', int(SAMPLE_RATE * DURATION))
val_dataset = EmotionDatasetRaw(train_df.iloc[val_idx], 'Russian Emotional Speech/train', int(SAMPLE_RATE * DURATION))

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [55]:
len(train_loader)

28

In [56]:
mfcc_transform = T.MFCC(
    sample_rate=SAMPLE_RATE,
    n_mfcc=N_MFCC,
    melkwargs={"n_fft": N_FFT, "hop_length": HOP_LENGTH, "n_mels": N_MELS}
).to(DEVICE)

In [57]:
input_dim = N_MFCC * (int(SAMPLE_RATE * DURATION / HOP_LENGTH) + 1)
hidden_sizes = [512, 256, 128, 64]
num_classes = len(EMOTIONS)

In [58]:
class EmotionClassifier(nn.Module):
    def __init__(self, input_dim, hidden_sizes, num_classes):
        super(EmotionClassifier, self).__init__()
        layers = []
        prev_dim = input_dim
        layers.append(nn.Flatten())
        for h_dim in hidden_sizes:
            layers.append(nn.Linear(prev_dim, h_dim))
            layers.append(nn.BatchNorm1d(h_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(0.1))
            prev_dim = h_dim
        layers.append(nn.Linear(prev_dim, num_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


In [59]:
model = EmotionClassifier(input_dim, hidden_sizes, num_classes).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [60]:
model

EmotionClassifier(
  (net): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=3760, out_features=512, bias=True)
    (2): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): ReLU()
    (4): Dropout(p=0.1, inplace=False)
    (5): Linear(in_features=512, out_features=256, bias=True)
    (6): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (7): ReLU()
    (8): Dropout(p=0.1, inplace=False)
    (9): Linear(in_features=256, out_features=128, bias=True)
    (10): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (11): ReLU()
    (12): Dropout(p=0.1, inplace=False)
    (13): Linear(in_features=128, out_features=64, bias=True)
    (14): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (15): ReLU()
    (16): Dropout(p=0.1, inplace=False)
    (17): Linear(in_features=64, out_features=7, bias=True)
  )
)

In [61]:
print("Начало обучения...")
best_val_acc = 0.0
for epoch in range(1, EPOCHS+1):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}")
    for waveforms, labels in pbar:
        waveforms, labels = waveforms.to(DEVICE), labels.to(DEVICE)
        # Извлекаем MFCC признаки на лету
        features = mfcc_transform(waveforms)

        optimizer.zero_grad()
        outputs = model(features)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * waveforms.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        pbar.set_postfix(loss=loss.item(), acc=correct/total)

    train_acc = correct / total

    # Валидация
    model.eval()
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for waveforms, labels in val_loader:
            waveforms, labels = waveforms.to(DEVICE), labels.to(DEVICE)
            features = mfcc_transform(waveforms)
            outputs = model(features)
            _, preds = torch.max(outputs, 1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)
    val_acc = val_correct / val_total

    print(f"Epoch {epoch}/{EPOCHS} | Loss: {running_loss/len(train_loader):.4f} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_model.pth")
        print("  -> Сохранена лучшая модель")


Начало обучения...


Epoch 1/30:   0%|          | 0/28 [00:00<?, ?it/s]

Epoch 1/30 | Loss: 61.1312 | Train Acc: 0.1951 | Val Acc: 0.3304
  -> Сохранена лучшая модель


Epoch 2/30:   0%|          | 0/28 [00:00<?, ?it/s]

Epoch 2/30 | Loss: 51.9556 | Train Acc: 0.4002 | Val Acc: 0.3170


...

Epoch 28/30 | Loss: 1.7890 | Train Acc: 0.9877 | Val Acc: 0.3527


Epoch 29/30:   0%|          | 0/28 [00:00<?, ?it/s]

Epoch 29/30 | Loss: 2.7231 | Train Acc: 0.9753 | Val Acc: 0.4152


Epoch 30/30:   0%|          | 0/28 [00:00<?, ?it/s]

Epoch 30/30 | Loss: 2.1450 | Train Acc: 0.9798 | Val Acc: 0.3839


In [62]:
print("Загрузка test.csv и извлечение признаков...")
test_df = pd.read_csv(TEST_CSV)

Загрузка test.csv и извлечение признаков...


In [63]:
class TestDataset(Dataset):
    def __init__(self, df, audio_dir, target_length):
        self.df = df.reset_index(drop=True)
        self.audio_dir = audio_dir
        self.target_length = target_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        file_path = os.path.join(self.audio_dir, row['path'])
        waveform, sr = torchaudio.load(file_path)
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)
        if waveform.shape[1] < self.target_length:
            pad_size = self.target_length - waveform.shape[1]
            waveform = torch.nn.functional.pad(waveform, (0, pad_size))
        else:
            waveform = waveform[:, :self.target_length]
        return waveform, idx


In [64]:
test_dataset = TestDataset(test_df, 'Russian Emotional Speech/test', int(SAMPLE_RATE * DURATION))
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

In [65]:
model.load_state_dict(torch.load("best_model.pth"))
model.eval()

EmotionClassifier(
  (net): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=3760, out_features=512, bias=True)
    (2): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): ReLU()
    (4): Dropout(p=0.1, inplace=False)
    (5): Linear(in_features=512, out_features=256, bias=True)
    (6): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (7): ReLU()
    (8): Dropout(p=0.1, inplace=False)
    (9): Linear(in_features=256, out_features=128, bias=True)
    (10): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (11): ReLU()
    (12): Dropout(p=0.1, inplace=False)
    (13): Linear(in_features=128, out_features=64, bias=True)
    (14): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (15): ReLU()
    (16): Dropout(p=0.1, inplace=False)
    (17): Linear(in_features=64, out_features=7, bias=True)
  )
)

In [66]:
all_preds = []
with torch.no_grad():
    for waveforms, indices in tqdm(test_loader, desc="Предсказание"):
        waveforms = waveforms.to(DEVICE)
        mfcc = mfcc_transform(waveforms)
        outputs = model(mfcc)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())

Предсказание:   0%|          | 0/280 [00:00<?, ?it/s]

In [74]:
submission = pd.DataFrame({
    'ID': np.arange(len(test_df)),
    'labels': all_preds
})
submission.to_csv("submission.csv", index=False)
print("Сабмишн сохранён в submission.csv")
print("Первые 5 строк:")
print(submission.head())

Сабмишн сохранён в submission.csv
Первые 5 строк:
   ID  labels
0   0       2
1   1       0
2   2       3
3   3       4
4   4       2
